# lorafusion: Triton GPU Kernel Verification

Run this notebook on a Colab **GPU runtime** (`Runtime > Change runtime type > GPU`, T4 is fine) to compile and numerically validate `lorafusion/ops/triton_ops.py` against the CPU mock reference in `lorafusion/ops/mock_ops.py`, and to time the fused kernel against a naive baseline.

**Before running:** zip the project locally (`cd .. && zip -r lorafusion.zip lorafusion tests requirements.txt`) and have the zip ready to upload in the next cell.

**Every time you re-upload a new zip with kernel changes: `Runtime > Restart session` first, or better, close this tab and re-upload the .ipynb itself from disk if the notebook's own cells changed too.** Colab keeps `lorafusion.*` modules cached in memory across cells (`sys.modules`) -- overwriting the files on disk does NOT make Python re-read them. Cell 1 prints a hash of `triton_ops.py` so you can visually confirm the upload actually landed.

In [ ]:
import shutil, os
shutil.rmtree("lorafusion", ignore_errors=True)
shutil.rmtree("tests", ignore_errors=True)

from google.colab import files
uploaded = files.upload()  # select lorafusion.zip
import zipfile
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall(".")
os.remove(zip_name)

import hashlib
with open("lorafusion/ops/triton_ops.py", "rb") as f:
    print("triton_ops.py sha256:", hashlib.sha256(f.read()).hexdigest()[:12])
print("(compare this to the hash after your next edit -- if it's identical, the upload didn't pick up your change)")
os.listdir(".")

In [ ]:
!pip install -q -r requirements.txt triton pytest
import torch
assert torch.cuda.is_available(), "Select a GPU runtime: Runtime > Change runtime type > GPU"
print(torch.cuda.get_device_name(0))

## 1. Correctness: run the full test suite (CPU mock tests + GPU Triton tests)

In [ ]:
!python -m pytest tests/ -v

If `test_triton_ops.py` fails, copy the full traceback back to Claude Code -- the kernel source lives in `lorafusion/ops/triton_ops.py` and can be fixed locally, then re-zipped. **Restart the session before re-uploading** (see warning above).

## 2. Performance: reproducing the paper's Claim C2 (Figure 17)

**Read this before interpreting the numbers.** The paper reports two very different families of speedup, and they are not comparable:

| Claim | Number | What it measures | Reproducible here? |
|---|---|---|---|
| **C1** (Fig. 14) | 1.96x peak / 1.47x avg vs Megatron-LM | **End-to-end training throughput**, 4x H100, FSDP + pipeline parallelism + multi-adapter job scheduling | No — needs 4x H100 |
| combined | 2.05x peak | C1 + adaptive scheduling together | No |
| **C2** (Fig. 17) | **1.39x peak / 1.27x avg** | **The FusedLoRA kernel alone**, vs the standard Torch LoRA implementation | **Yes — this cell** |

`triton_ops.py` implements the C2 kernel. **1.27x-1.39x is its ceiling**, not 1.96x.

Most of C1 comes from the *scheduler*, not the kernel: grouping adapters cuts the pipeline bubble ratio from 48.79% to 11.09% (Section 6.5). That half of the system lives in `lorafusion/scheduler/` and is already covered by `test_scheduler.py`.

Two caveats for a Colab T4:
1. The baseline must be **the same precision as the kernel**. The paper compares against Torch LoRA in half precision on H100. Comparing our fp16 kernel to an fp32 baseline measures fp16-vs-fp32, not fusion.
2. The paper is explicit that this hardware will under-deliver: *"Systems with higher compute-to-memory-bandwidth ratios typically yield superior performance, while older hardware with lower ratios may exhibit reduced performance gains."* The gain comes from saving DRAM traffic on activations, so it scales with the compute-to-bandwidth ratio (T4 ~203 FLOP/byte vs H100 ~295). Expect below 1.27x on a T4 even when the kernel is correct.


In [ ]:
import time
import torch
from lorafusion.ops import triton_ops
from lorafusion.ops.config import AdapterSpec, TileRoute, TileRoutingConfig

device = "cuda"
torch.manual_seed(0)
DT = torch.float16  # half precision, matching the paper's setup

def torch_lora(x, w, a, b, scaling):
    """The paper's baseline: 'the standard Torch LoRA implementation'.

    Note the memory traffic this costs on the full-sized (m,n) tensors:
    one write of y, one write of delta, then a read-modify-write of y.
    Eliminating that is exactly what FusedLoRA's operation 2 is for.
    """
    y = x @ w.t()
    y += scaling * ((x @ a.t()) @ b.t())
    return y

def bench(fn, *args, iters=30):
    for _ in range(10):
        fn(*args)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters):
        fn(*args)
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / iters

# LLaMa-3.1-8B projection shapes (hidden 4096, intermediate 14336), the model
# family Figure 17 sweeps, at a range of token counts per micro-batch.
LAYERS = [
    ("q_proj",    4096,  4096),
    ("gate_proj", 4096, 14336),
]
RANK = 16

print(f"{'layer':>10} {'tokens':>7} {'torch':>9} {'fused':>9} {'speedup':>8} "
      f"{'TFLOP/s':>8} {'traffic saved':>13}  err")
speedups = []
for name, k_in, n_out in LAYERS:
    for tokens in [4096]:
        x = torch.randn(tokens, k_in, device=device, dtype=DT)
        w = torch.randn(n_out, k_in, device=device, dtype=DT)
        a = torch.randn(RANK, k_in, device=device, dtype=DT) * 0.02
        b = torch.randn(n_out, RANK, device=device, dtype=DT) * 0.02
        scaling = 2.0

        adapters = {0: AdapterSpec(0, RANK, RANK * scaling, k_in, n_out)}
        routing = TileRoutingConfig(tile_size=tokens, routes=[TileRoute(0, tokens, 0)])
        fused_args = (x, w, {0: a}, {0: b}, adapters, routing)

        t_torch = bench(torch_lora, x, w, a, b, scaling)
        t_fused = bench(triton_ops.fused_lora_forward, *fused_args)

        ref = torch_lora(x.float(), w.float(), a.float(), b.float(), scaling)
        got = triton_ops.fused_lora_forward(*fused_args).float()
        err = ((got - ref).abs().max() / ref.abs().max()).item()

        sp = t_torch / t_fused
        speedups.append(sp)
        tflops = 2 * tokens * k_in * n_out / t_fused / 1e12
        # (m,n) tensor traffic the fusion removes: baseline writes y, writes
        # delta, then reads both and writes y again; fused writes y once.
        saved = 4 * tokens * n_out * x.element_size() / 1e6
        print(f"{name:>10} {tokens:7d} {t_torch*1e3:8.3f}m {t_fused*1e3:8.3f}m "
              f"{sp:7.2f}x {tflops:8.1f} {saved:11.1f}MB  {err:.1e}")

avg, peak = sum(speedups)/len(speedups), max(speedups)
print(f"\nFusedLoRA forward: average {avg:.2f}x, peak {peak:.2f}x")
print(f"Paper C2 (Fig 17, H100):    average 1.27x, peak 1.39x")
print("\nIf you are on a T4, landing under 1.27x is expected and is stated in the")
print("paper (Sec. A.2.2). What matters is that this is >1.0x at equal precision.")
print("The 1.96x / 2.05x numbers are end-to-end system results on 4xH100 and are")
print("dominated by the job scheduler (lorafusion/scheduler/), not by this kernel.")


## 3. Diagnostic: is a BARE Triton fp16 matmul (no fusion, no branching) even close to cuBLAS here?

This isolates the variable: if a minimal, textbook Triton matmul (straight from Triton's own tutorial, no LoRA fusion, no `tl.trans` branching, no tile-routing) is ALSO far from cuBLAS on this GPU/Triton version, the gap is environmental and not really fixable by us. If it's close to cuBLAS, the problem is specific to our fused kernel's structure and worth debugging further.

In [ ]:
import triton
import triton.language as tl

@triton.autotune(
    configs=[
        triton.Config({"BLOCK_M": 64, "BLOCK_N": 64, "BLOCK_K": 32}, num_warps=4, num_stages=2),
        triton.Config({"BLOCK_M": 128, "BLOCK_N": 128, "BLOCK_K": 32}, num_warps=8, num_stages=3),
        triton.Config({"BLOCK_M": 128, "BLOCK_N": 64, "BLOCK_K": 32}, num_warps=4, num_stages=4),
        triton.Config({"BLOCK_M": 64, "BLOCK_N": 128, "BLOCK_K": 64}, num_warps=4, num_stages=4),
    ],
    key=["M", "N", "K"],
)
@triton.jit
def bare_matmul_kernel(
    a_ptr, b_ptr, c_ptr, M, N, K,
    stride_am, stride_ak, stride_bk, stride_bn, stride_cm, stride_cn,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    for k in range(0, K, BLOCK_K):
        k_idx = offs_k + k
        a_tile = tl.load(a_ptr + offs_m[:, None] * stride_am + k_idx[None, :] * stride_ak,
                          mask=(offs_m[:, None] < M) & (k_idx[None, :] < K), other=0.0).to(tl.float16)
        b_tile = tl.load(b_ptr + k_idx[:, None] * stride_bk + offs_n[None, :] * stride_bn,
                          mask=(k_idx[:, None] < K) & (offs_n[None, :] < N), other=0.0).to(tl.float16)
        acc += tl.dot(a_tile, b_tile)
    c_mask = (offs_m[:, None] < M) & (offs_n[None, :] < N)
    tl.store(c_ptr + offs_m[:, None] * stride_cm + offs_n[None, :] * stride_cn, acc, mask=c_mask)

def bare_matmul(a, b):
    M, K = a.shape
    K2, N = b.shape
    c = torch.empty((M, N), device=a.device, dtype=torch.float32)
    grid = lambda META: (triton.cdiv(M, META["BLOCK_M"]), triton.cdiv(N, META["BLOCK_N"]))
    bare_matmul_kernel[grid](
        a, b, c, M, N, K,
        a.stride(0), a.stride(1), b.stride(0), b.stride(1), c.stride(0), c.stride(1),
    )
    return c

M, K, N = 1024, 4096, 4096
a = torch.randn(M, K, device=device, dtype=torch.float32)
b = torch.randn(K, N, device=device, dtype=torch.float32)

bare_t = bench(bare_matmul, a, b)
cublas_t = bench(lambda a, b: a @ b, a, b)
print(f"bare Triton fp16 matmul: {bare_t*1e3:.3f}ms   cuBLAS: {cublas_t*1e3:.3f}ms   ratio(bare/cublas)={bare_t/cublas_t:.2f}x")
print("best config picked:", bare_matmul_kernel.best_config if hasattr(bare_matmul_kernel, 'best_config') else "(call once more with the same shape to populate this)")